# Tutorial 3 — Operators and Gradients

The denotation of a formula can be computed with different fuzzy operators. Two operators may give similar truth degrees and still **train very differently**, because training depends on the gradient that each operator passes to the predicates and to the other learnable symbols.

Section 1 compares four implications on the same formula and shows which individuals receive a gradient under each. Section 2 shows how the exponent `p` of the universal quantifier distributes the gradient among the individuals. Section 3 plots every connective over $[0,1]^2$, either its value or the magnitude of its gradient, with sliders for `p` and for the stability parameter $\varepsilon$.

> **Dependencies of section 3:** `matplotlib` and `ipywidgets`, installed with `pip install sltn[notebooks]`.

In [ ]:
import torch
import torch.nn as nn
import sltn
from sltn import Signature, Interpretation, Logic
from sltn.interpretation.operators import aggregator, implication

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

## Setup

An animal is a vector in ℝ² of two features, its size and its furriness, as in Tutorials 1 and 2. Both predicates are now **trainable networks**, so that the gradient each one receives can be inspected.

`Net` stores its last output and calls `retain_grad` on it. After a backward pass, `net.out.grad` therefore contains $\partial\,\mathrm{sat} / \partial\, \mathrm{IsDog}(x_i)$, the change in the satisfaction caused by a change in the output of the network for animal $i$. This is needed only for inspection, not for training.

`make_interp(logic)` builds an interpretation with the given `Logic`, or the default one for `None`, and always with the same networks and data. The experiments below therefore differ only in their operators.

In [ ]:
sig = Signature("animals")
sig.sort("Animal")
sig.variable("x", "Animal")
sig.predicate("IsDog", ["Animal"])
sig.predicate("IsLarge", ["Animal"])


class Net(nn.Module):
    """A small predicate network that keeps its last output, to read its gradient."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 8), nn.ELU(), nn.Linear(8, 1), nn.Sigmoid())

    def forward(self, a):                 # a: [B, 2]
        self.out = self.net(a)            # [B, 1]
        self.out.retain_grad()
        return self.out


is_dog, is_large = Net(), Net()
animals = torch.randn(16, 2, device=device)


def make_interp(logic=None):
    interp = Interpretation(sig, logic=logic).to(device)
    interp["Animal"] = sltn.Type("R2", shape=(2,))
    interp["x"] = animals
    interp["IsDog"] = is_dog
    interp["IsLarge"] = is_large
    return interp


def output_grad(net):
    """The gradient of the last backward pass w.r.t. the network's output, one value per animal."""
    return net.out.grad[:, 0] if net.out.grad is not None else torch.zeros(len(animals), device=device)

## 1 — Four implications, four learning signals

The formula is "every dog is large", `∀ x: (IsDog(x) → IsLarge(x))`. With $a = \mathrm{IsDog}(x)$ and $b = \mathrm{IsLarge}(x)$:

| Implication | $a \to b$ | Where it is 1 |
|---|---|---|
| **Reichenbach** (the default, as in LTN) | $1 - a + a b$ | only at $a = 0$ or $b = 1$ |
| **Goguen** | 1 if $a \le b$, else $b / a$ | $a \le b$ |
| **Łukasiewicz** | $\min(1,\ 1 - a + b)$ | $a \le b$ |
| **Gödel** | 1 if $a \le b$, else $b$ | $a \le b$ |

The `Logic` is passed to the interpretation with `logic=`. Each configuration below changes only the implication, and `∀` remains pMeanError with p = 2.

In [ ]:
phi = sig.parse("forall x: (IsDog(x) -> IsLarge(x))")

configs = {
    "Reichenbach (default)": None,
    "Goguen":                Logic(Implies=implication.ImpliesGoguen()),
    "Łukasiewicz":           Logic(Implies=implication.ImpliesLuk()),
    "Gödel":                 Logic(Implies=implication.ImpliesGodel()),
}

grads = {}
print(f"{'implication':<24s} {'sat':>6s}   no gradient   mean |∂sat/∂IsLarge|: a ≤ b    a > b")
for name, logic in configs.items():
    interp = make_interp(logic)
    sat = interp(phi).value
    sat.backward()

    g_dog, g_large = output_grad(is_dog), output_grad(is_large)
    grads[name] = g_large.clone()
    holds = (is_dog.out <= is_large.out)[:, 0]               # animals where a <= b
    no_grad = ((g_dog.abs() < 1e-6) & (g_large.abs() < 1e-6)).sum().item()
    print(f"{name:<24s} {sat.item():6.3f}   {no_grad:2d} of {len(animals)}"
          f"      {g_large[holds].abs().mean().item():22.4f}  {g_large[~holds].abs().mean().item():8.4f}")
    interp.zero_grad()

print(f"\n{holds.sum().item()} of {len(animals)} animals have IsDog(x) <= IsLarge(x)")

The table shows the following.

- **Goguen and Gödel give no gradient to the animals with $a \le b$.** For these animals the implication equals 1, and it remains 1 under small changes of $a$ or $b$. The networks therefore receive no signal from them, and all the gradient comes from the animals with $a > b$.
- **Reichenbach gives a gradient to every animal.** Since $1 - a + ab$ equals 1 only at $a = 0$ or $b = 1$, it decreases $a$ and increases $b$ also where $a \le b$. The signal is denser, but it also changes animals that already satisfy the formula.
- **Łukasiewicz** gives no gradient where $a \le b$, as Goguen does. Where $a > b$, the gradients with respect to $a$ and $b$ have the same magnitude and opposite signs.
- The truth degrees differ less than the gradients. **Gödel**, moreover, gives no gradient to `IsDog`, since where $a > b$ its value is $b$, which does not depend on $a$.

The following cell lists, for the first 6 animals, the gradient with respect to `IsLarge` under Reichenbach and Goguen.

In [ ]:
a, b = is_dog.out.detach()[:, 0], is_large.out.detach()[:, 0]
print("animal   IsDog  IsLarge   a <= b    ∂sat/∂IsLarge: Reichenbach   Goguen")
for i in range(6):
    print(f"{i:>6d}   {a[i]:.3f}    {b[i]:.3f}   {str(bool(a[i] <= b[i])):>6s}"
          f"   {grads['Reichenbach (default)'][i].item():26.4f}   {grads['Goguen'][i].item():8.4f}")

## 2 — The exponent `p` of ∀

`∀` is pMeanError, $1 - \big(\mathrm{mean}_i\, (1 - a_i)^p\big)^{1/p}$. Its gradient with respect to individual $i$ is proportional to $(1 - a_i)^{p-1}$. An individual with a lower truth degree therefore receives a larger share of the gradient, and `p` determines how much larger.

- With $p = 1$, ∀ is the arithmetic mean, and every individual receives the same share.
- A larger $p$ brings the value closer to the minimum, and concentrates the gradient on the individuals with the lowest truth degrees. As $p \to \infty$, only the lowest one receives a gradient.

The following cell computes, for `∀ x: IsDog(x)`, the value and the share of the gradient received by the 4 animals with the lowest truth degree of `IsDog` among the 16. With $p = 1$ this share is 4/16 = 25%.

In [ ]:
interp = make_interp()
psi = sig.parse("forall x: IsDog(x)")

for p in [1, 2, 4, 10]:
    interp.logic["∀"] = aggregator.AggregPMeanError(p=p)
    sat = interp(psi).value
    sat.backward()

    g = output_grad(is_dog)
    worst = is_dog.out.detach()[:, 0].argsort()[:4]          # the 4 animals with the lowest IsDog
    print(f"p={p:2d}   ∀ x: IsDog(x) = {sat.item():.4f}   gradient share of the 4 worst: {(g[worst].sum() / g.sum()).item():.0%}")
    interp.zero_grad()

The default $p = 2$ is a compromise. The individuals with low truth degrees receive more gradient, but the others still receive some. A large $p$ concentrates training on a few individuals, which enforces a rule more strictly but makes training noisier.

## 3 — Connectives as surfaces

The following cell plots a connective over the unit square $[0,1]^2$, with `Input 1` as its first argument and `Input 2` as its second. **Show** selects the plotted quantity.

- **value:** the truth degree, from 0 (dark) to 1 (light), with a contour line every 0.05.
- **|gradient|:** the magnitude of the gradient, $\log_{10} \lVert \nabla f(x, y) \rVert$. In the darkest regions, below $10^{-3}$, the operator is **flat**, and an input there receives almost no gradient. In the lightest regions, a small change of the input gives a large gradient. The contour lines are half a decade apart, a factor of about 3.

The plots show the following.

- **Flat regions.** The Goguen, Gödel and Łukasiewicz implications are flat above the diagonal, $x \le y$, as in section 1. `And (Min)` passes a gradient to only one of its inputs at each point.
- **Steep regions.** Goguen is steep near $x = 0$ below the diagonal, where $y / x$ changes rapidly.
- **The slider p.** For the p-means, a larger $p$ moves the surface towards the maximum (`p-Mean`) or the minimum (`p-Mean Error`).
- **The slider ε.** Stable operators keep their inputs at least $\varepsilon$ away from 0 and 1. A large $\varepsilon$ changes the values visibly. The default is $10^{-6}$.

The two p-means are aggregators. They are applied here to the two values `torch.stack([x, y])`, as a quantifier aggregates two individuals.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, FloatSlider, FloatLogSlider
import sltn.interpretation.operators as ops

In [ ]:
def make_operator(name, p, eps):
    """The selected connective, as a function of two tensors."""
    pair = lambda agg: (lambda x, y: agg(torch.stack(torch.broadcast_tensors(x, y)), dim=[0]))
    return {
        "And (Prod)":               ops.AndProd(eps=eps),
        "And (Min)":                ops.AndMin(),
        "And (Łukasiewicz)":        ops.AndLuk(eps=eps),
        "And (pMean)":              ops.AndPMean(p=p, eps=eps),
        "Or (ProbSum)":             ops.OrProbSum(eps=eps),
        "Or (Max)":                 ops.OrMax(),
        "Implies (Goguen)":         ops.ImpliesGoguen(eps=eps),
        "Implies (Reichenbach)":    ops.ImpliesReichenbach(eps=eps),
        "Implies (Łukasiewicz)":    ops.ImpliesLuk(eps=eps),
        "Implies (Gödel)":          ops.ImpliesGodel(),
        "Implies (Kleene-Dienes)":  ops.ImpliesKleeneDienes(),
        "Equiv (Prod + Reichenbach)": ops.Equiv(and_op=ops.AndProd(eps=eps), implies_op=ops.ImpliesReichenbach(eps=eps)),
        "Equiv (Łukasiewicz)":      ops.Equiv(and_op=ops.AndLuk(eps=eps), implies_op=ops.ImpliesLuk(eps=eps)),
        "Equiv (Similarity)":       ops.EquivSimilarity(p=p, eps=eps),
        "p-Mean (∃)":               pair(ops.AggregPMean(p=p, eps=eps)),
        "p-Mean Error (∀)":         pair(ops.AggregPMeanError(p=p, eps=eps)),
    }[name]


def plot_connective(name, show, p, eps):
    grid = torch.linspace(0, 1, 101)
    X, Y = torch.meshgrid(grid, grid, indexing="xy")        # X: Input 1 (columns), Y: Input 2 (rows)
    X.requires_grad_(True)
    Y.requires_grad_(True)
    Z = make_operator(name, p, eps)(X, Y)

    if show == "value":
        data = Z.detach()
        levels, lines, ticks, label = np.linspace(0, 1, 101), np.linspace(0, 1, 21), np.linspace(0, 1, 6), "truth value"
    else:
        gx, gy = torch.autograd.grad(Z.sum(), [X, Y], allow_unused=True)
        gx = torch.zeros_like(X) if gx is None else gx
        gy = torch.zeros_like(Y) if gy is None else gy
        data = torch.log10(torch.sqrt(gx ** 2 + gy ** 2).clamp_min(1e-3)).clamp_max(2)
        levels, lines, ticks, label = np.linspace(-3, 2, 101), np.linspace(-3, 2, 11), np.arange(-3, 3), "log10 |gradient|"

    fig, ax = plt.subplots(figsize=(6, 5))
    contour = ax.contourf(grid.numpy(), grid.numpy(), data.numpy(), levels=levels, cmap="viridis")
    ax.contour(grid.numpy(), grid.numpy(), data.numpy(), levels=lines, colors="k", linewidths=0.5)
    fig.colorbar(contour, ax=ax, ticks=ticks, label=label)
    ax.set_title(f"{name}   (p={p:g}, ε={eps:.0e})")
    ax.set_xlabel("Input 1 (x)")
    ax.set_ylabel("Input 2 (y)")
    ax.set_aspect("equal")
    plt.tight_layout()
    plt.show()


interact(
    plot_connective,
    name=Dropdown(options=[
        "And (Prod)", "And (Min)", "And (Łukasiewicz)", "And (pMean)",
        "Or (ProbSum)", "Or (Max)",
        "Implies (Goguen)", "Implies (Reichenbach)", "Implies (Łukasiewicz)",
        "Implies (Gödel)", "Implies (Kleene-Dienes)",
        "Equiv (Prod + Reichenbach)", "Equiv (Łukasiewicz)", "Equiv (Similarity)",
        "p-Mean (∃)", "p-Mean Error (∀)",
    ], value="Implies (Reichenbach)", description="Connective:"),
    show=Dropdown(options=["value", "|gradient|"], value="|gradient|", description="Show:"),
    p=FloatSlider(min=0.5, max=10.0, step=0.5, value=2.0, description="p:", continuous_update=False),
    eps=FloatLogSlider(base=10, min=-8, max=-1, step=0.5, value=1e-6, description="ε:", continuous_update=False),
);

**The effect of `stable`.** With `stable=True`, the default for every operator, the inputs are kept at least $\varepsilon$ away from 0 and 1 before the operator is applied. This prevents **infinite or undefined gradients at exactly 0 and 1**, for instance for pMeanError when every individual has truth degree 1, or for Goguen at $x = 0$. It does **not** remove flat regions. Goguen above the diagonal and `And (Min)` remain flat, because flatness is a property of the operator itself. A gradient in such a region requires another operator (section 1).

---

**Next:** [Tutorial 4 — KB & Learning](4-KB&Learning.ipynb) trains predicates on a knowledge base of formulas.